Training Set

Phase 3: Learning Constitutive Laws from Granular Flow Data

Project: Neural Network Framework for Learning Constitutive Laws in 2D Granular Flow
Author: Abhishek Tagalpallewar

Description:
This notebook implements the complete neural network training pipeline using
Discrete Element Method (DEM) simulation data.

The workflow includes:

• Loading and preprocessing granular flow datasets.
• Identifying quasi-linear regions suitable for constitutive modeling.
• Applying Reference Normalization for scale invariance.
• Performing data augmentation using Gaussian noise.
• Training multiple neural network architectures.
• Conducting a hyperparameter tournament.
• Selecting the best-performing constitutive model.

By the end of this notebook, a trained model capable of predicting
σxx, σxy and σyy from velocity gradient information is obtained.

1. Load DEM Simulation Data

2. Quasi-linear Region Detection

3. Reference Normalization

4. Data Augmentation

5. Train / Validation Split

6. Neural Network Architecture

7. Hyperparameter Tournament

8. Model Evaluation

9. Leaderboard of Best Models

In [1]:
"""
Phase 3: Advanced Constitutive Training - Normalization & Augmentation
Project: Learning Constitutive Laws in 2D Granular Flow
Author: Abhishek Tagalpallewar

Description:
Final training pipeline integrating Reference Normalization for scale-invariance 
and Data Augmentation for model robustness. Identifies the 
high-fidelity champion model through an architectural tournament.
"""

import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import os

# --- 1. DATA LOADING & HORIZONTAL AVERAGING ---
basePath = '/home/abhishek/granular_data'
def load_and_avg(name):
    path = os.path.join(basePath, name + '.txt')
    if not os.path.exists(path): return np.zeros(600)
    data = np.loadtxt(path, delimiter=',')
    return np.mean(data, axis=0) # Reduces particle-level noise to bulk trends[cite: 1, 3]

v_grad_prof = load_and_avg('Vx_grad_lucy_slice')
sxx_prof    = load_and_avg('Sigma_xx_lucy_slice')
sxy_prof    = load_and_avg('Sigma_xy_lucy_slice')
syy_prof    = load_and_avg('Sigma_yy_lucy_slice')
z_norm      = np.loadtxt(os.path.join(basePath, 'z_norm_slice.txt'), delimiter=',')

# --- 2. QUASI-LINEAR MASKING (Stability Detection) ---
def get_quasi_linear_mask(profile, window=25, percentile=60):
    slopes    = np.gradient(profile, z_norm)
    local_std = pd.Series(slopes).rolling(window=window, center=True).std().bfill().ffill().values
    return local_std < np.percentile(local_std, percentile) # Isolate stable flow regions[cite: 1, 3]

linearity_mask = get_quasi_linear_mask(sxx_prof) & get_quasi_linear_mask(sxy_prof) & get_quasi_linear_mask(syy_prof)
wall_mask      = np.abs(v_grad_prof) > 0.3 # Filter boundary layer interference[cite: 1]
idx            = np.where(linearity_mask & wall_mask)[0]

# --- 3. REFERENCE NORMALIZATION & AUGMENTATION ---
X_raw = v_grad_prof[idx].reshape(-1, 1)
Y_raw = np.column_stack((sxx_prof[idx], sxy_prof[idx], syy_prof[idx]))

# Breakthrough: Scale-invariance via Reference Normalization[cite: 1]
ref_vgrad_train  = np.mean(np.abs(X_raw))
ref_stress_train = np.mean(np.abs(Y_raw))

X_raw_norm = X_raw / ref_vgrad_train
Y_raw_norm = Y_raw / ref_stress_train

# Breakthrough: Gaussian noise injection to double training points[cite: 1]
np.random.seed(42)
noise_x = np.random.normal(0, 0.01, X_raw_norm.shape)
noise_y = np.random.normal(0, 0.01, Y_raw_norm.shape)
X_aug   = np.vstack([X_raw_norm, X_raw_norm + noise_x])
Y_aug   = np.vstack([Y_raw_norm, Y_raw_norm + noise_y])

# --- 4. DATA PREP & SCALING ---
scaler_x, scaler_y = StandardScaler(), StandardScaler()
X_scaled = scaler_x.fit_transform(X_aug)
Y_scaled = scaler_y.fit_transform(Y_aug)

X_train, X_val, Y_train, Y_val = train_test_split(X_scaled, Y_scaled, test_size=0.2, random_state=42)
X_train_t, Y_train_t = torch.FloatTensor(X_train), torch.FloatTensor(Y_train)
X_val_t, Y_val_t     = torch.FloatTensor(X_val), torch.FloatTensor(Y_val)

# --- 5. TOURNAMENT ARCHITECTURE ---
class MathStrengthNet(nn.Module):
    def __init__(self, act_fn):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 128), act_fn,
            nn.Linear(128, 64), act_fn,
            nn.Linear(64, 3) # Output: sigma_xx, sigma_xy, sigma_yy[cite: 1, 3]
        )
    def forward(self, x): return self.net(x)

def get_loss_fn(idx):
    if idx == 1: return nn.MSELoss()
    if idx == 2: return nn.L1Loss()
    if idx == 3: return nn.HuberLoss()
    if idx == 4: return lambda p, t: torch.mean((torch.log1p(torch.abs(p)) - torch.log1p(torch.abs(t)))**2)
    return lambda p, t: torch.mean(torch.abs(p - t) / (torch.abs(t) + 1e-8))

act_names = {1: 'ReLU', 2: 'Tanh', 3: 'Swish', 4: 'Mish', 5: 'GELU'}
loss_names = {1: 'MSE', 2: 'MAE', 3: 'Huber', 4: 'MSLE', 5: 'RE'}
benchmark_results = []
trained_models = {}

# --- 6. TOURNAMENT EXECUTION ---
print(f"🚀 Starting Tournament on {len(X_aug)} points...")
for a_idx, a_fn in {1: nn.ReLU(), 2: nn.Tanh(), 3: nn.SiLU(), 4: nn.Mish(), 5: nn.GELU()}.items():
    for l_idx in range(1, 6):
        model = MathStrengthNet(a_fn)
        optimizer = optim.AdamW(model.parameters(), lr=0.001)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=3000)
        criterion = get_loss_fn(l_idx)
        
        hist_re = []
        for epoch in range(3000):
            model.train(); optimizer.zero_grad()
            preds = model(X_train_t)
            l = criterion(preds, Y_train_t)
            l.backward(); optimizer.step(); scheduler.step()
            with torch.no_grad():
                re_step = torch.mean(torch.abs(preds - Y_train_t) / (torch.abs(Y_train_t) + 1e-8)).item()
                hist_re.append(re_step)
        
        model.eval()
        with torch.no_grad():
            preds_v = model(X_val_t)
            re_val = torch.mean(torch.abs(preds_v - Y_val_t) / (torch.abs(Y_val_t) + 1e-8)).item()
        
        combo = f"{act_names[a_idx]} + {loss_names[l_idx]}"
        benchmark_results.append({'Combo': combo, 'RE': re_val, 'History': hist_re})
        trained_models[combo] = model

# --- 7. RESULTS & INTERACTIVE PLOT ---
df = pd.DataFrame(benchmark_results).sort_values('RE')
print("\n🏆 LEADERBOARD (Top 5):")
print(df[['Combo', 'RE']].head(5).to_string(index=False))

print("\n🚀 Ready for Visual Inspection. Select a combination to view plots.")
while True:
    act_in = input("\nSelect Activation (e.g., Tanh) or 'exit': ").strip()
    if act_in.lower() == 'exit': break
    loss_in = input("Select Loss (e.g., Huber): ").strip()
    target = f"{act_in} + {loss_in}"

    if target in trained_models:
        res = df[df['Combo'] == target].iloc[0]
        model = trained_models[target]
        model.eval()
        with torch.no_grad():
            sort_idx = np.argsort(X_raw.flatten())
            X_plot_t = torch.FloatTensor(scaler_x.transform(X_raw[sort_idx] / ref_vgrad_train))
            preds = scaler_y.inverse_transform(model(X_plot_t).numpy()) * ref_stress_train
            
        fig, axs = plt.subplots(2, 2, figsize=(12, 8))
        fig.suptitle(f"Constitutive Law: {target}", fontsize=14)
        titles = [r'$\sigma_{xx}$', r'$\sigma_{xy}$', r'$\sigma_{yy}$']
        for i in range(3):
            ax = axs[i//2, i%2]
            ax.scatter(X_raw, Y_raw[:, i], color='gray', alpha=0.3, label='True Data')
            ax.plot(X_raw[sort_idx], preds[:, i], 'r-', label='Prediction')
            ax.set_title(titles[i]); ax.legend()
        
        ax_hist = axs[1, 1]
        ax_hist.plot(res['History'], color='teal')
        ax_hist.set_yscale('log'); ax_hist.set_title("Training Convergence"); plt.show()
    else:
        print("Combo not found.")

FileNotFoundError: /home/abhishek/granular_data\z_norm_slice.txt not found.